<a href="https://colab.research.google.com/github/kookie-707/starzplay-internship/blob/main/RAG-chatbot/RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain openai faiss-cpu python-dotenv
!pip install -U langchain-community


In [ ]:
!pip install -q transformers

In [ ]:
!pip install pinecone

In [ ]:
import os
from langchain.document_loaders import CSVLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from dotenv import load_dotenv
from langchain.schema import Document
import pandas as pd
from transformers import GPT2TokenizerFast
from getpass import getpass
from openai import OpenAI
import pinecone
from langchain.vectorstores import Pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
csv_path = "/content/drive/Shared drives/Starzplay/starzplay_faq.csv"

df = pd.read_csv(csv_path)

# Combine tab + question + answer into single doc text
docs = []
for _, row in df.iterrows():
    content = f"[{row['tab']}]\nQ: {row['question']}\nA: {row['answer']}"
    doc = Document(page_content=content)
    docs.append(doc)

print(f"Loaded {len(docs)} FAQ entries")


In [ ]:

faq_entries = [
    f"[{row['tab']}] Q: {row['question']} A: {row['answer']}"
    for _, row in df.iterrows()
]
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
token_counts = [len(tokenizer.encode(entry)) for entry in faq_entries]

# Summarize
total_tokens = sum(token_counts)
average_tokens = total_tokens / len(token_counts)
max_tokens = max(token_counts)

print(f"Total FAQ entries: {len(faq_entries)}")
print(f"Total tokens: {total_tokens}")
print(f"Avg tokens per entry: {average_tokens:.2f}")
print(f"Max tokens in one entry: {max_tokens}")


In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()

In [ ]:
models = client.models.list()
chat_models = []
embedding_models = []

for model in models.data:
    model_id = model.id
    if "gpt" in model_id:
        chat_models.append(model_id)
    elif "embedding" in model_id:
        embedding_models.append(model_id)

print("\nAvailable Chat Models:")
for m in chat_models:
    print(" -", m)

print("\nAvailable Embedding Models:")
for m in embedding_models:
    print(" -", m)

In [ ]:
pc = Pinecone(api_key="")
index_name = "starzplay-faqs"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index(index_name)

# 3) Local embeddings using HuggingFace
embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim
texts     = [d.page_content for d in docs]
vectors   = embed_model.encode(texts).tolist()

# 4) Pinecone for vector storage
upserts = [
    {
      "id":       f"faq-{i}",
      "values":   vectors[i],
      "metadata": {"chunk_text": texts[i]}
    }
    for i in range(len(texts))
]

index.upsert(vectors=upserts)

print(f"Upserted {len(upserts)} FAQ entries into `{index_name}`")


In [ ]:
def retrieve_faqs(query: str, top_k: int = 5) -> list[str]:

    q_vec = embed_model.encode([query])[0].tolist()
    res = index.query(
        vector=q_vec,
        top_k=top_k,
        include_metadata=True
    )
    return [m["metadata"]["chunk_text"] for m in res["matches"]]

SYSTEM_PROMPT = """
You are a helpful and polite support agent for StarzPlay.
You can answer any question related to StarzPlay, including:

- Subscription and billing
- Available content (movies, series)
- Supported devices
- Account management
- Features like subtitles, languages, streaming, free tv, store, channels
- About starzplay and the device to watch starzplay on


Please stick to only info you have access to and don't make up answers or hallucinate with no evidence.
if asked and you don't have a trustworthy, evidence-based answer, tell them you do not have access to that info right now but maybe in the future when i am trained further!
Do not give info or recommendations unless you have direct access to the information.

Only when the user asks something that is clearly unrelated to StarzPlay (like world news, weather, science, etc.), politely say:

"I'm sorry, I can only answer questions related to StarzPlay's Services."

Always try to be helpful if the question is even slightly relevant.
Also only answer in english, if user asks a question in a different language you have to tell them that you only speak English and answer English queries.
"""

def answer_query(query: str) -> str:
    faqs = retrieve_faqs(query, top_k=5)
    context = "\n\n".join(faqs)
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": f"""Context: {context}
         Question: {query} Answer:"""}]

    resp = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        temperature=0
    )
    return resp.choices[0].message.content.strip()

In [ ]:

print("StarzPlay FAQ Bot — type 'exit,' 'end' or 'quit' to end\n")

while True:
    user_input = input("User: ").strip()
    if user_input.lower() in ("exit", "quit", "end"):
        print("The End!")
        break

    # Fetch answer
    try:
        response = answer_query(user_input)
        print("FAQ Agent:", response, "\n")
    except Exception as e:
        print("Error generating response:", e, "\n")